# Phase 3 grade: every method on every clean line (step 7)
This is the first Phase 3 code that reads actual development after 2007. The estimates were built on known cells only by `src/backtest.py` and frozen in commit `acd7cd9`, tag `phase3-freeze`, with their SHA-256 hashes in `notes/phase3_freeze.md`.

Grading rules (set before the freeze; project decision log and Phase 3 doc):
- **Matching basis:** paid methods against actual paid at 120 months, reported methods against actual reported at 120.
- **Unpaid** on both bases = losses at 120 − paid to date at 12/31/2007, so reported unpaid includes case still open at 120.
- **Error** = estimated unpaid − actual unpaid = estimate at 120 − actual at 120 (paid to date cancels). Positive = over-reserved.
- **Graded AYs:** 1999–2007 (D7). AY 1998 is already at 120 months.
- **Scales:** line totals as % of reserve (error ÷ actual unpaid, AYs 1999–2007 summed) and % of EP; AY-level rows as % of EP (% of reserve explodes on mature AYs, Phase 2).
- **Common sample (D4):** a line with a NaN or infinite estimate in any of the 40 combinations is left out of head-to-head comparisons; it stays in each method's own results.
- **Headline average (D3):** volume-weighted all years, for CL and for BF's CDFs.

One convention added here, before any actual is read: % of reserve is computed only where the line's actual unpaid total is > 0. A zero or negative denominator flips or blows up the sign; those lines are counted and still graded on % of EP.

In [1]:
# Step 7.1: load the frozen estimates and verify the hash
import sys; sys.path.insert(0, "..")
import hashlib, re
from pathlib import Path
import numpy as np
import pandas as pd
from src import backtest as bt
from src import methods as m

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
RESULTS = Path("../results"); DOCS = Path("../docs")
GRADED = range(1999, 2008)
EST = RESULTS / "estimates.csv"

if not EST.exists():  # gitignored (D5): rebuild with the frozen code
    _, _, est_built, d1 = bt.build()
    RESULTS.mkdir(exist_ok=True)
    est_built.to_csv(EST, index=False); d1.to_csv(RESULTS / "d1_counts.csv", index=False)

frozen = re.search(r"results/estimates.csv \| [\d,]+ \| ([0-9a-f]{64})",
                   Path("../notes/phase3_freeze.md").read_text()).group(1)
sha = hashlib.sha256(EST.read_bytes()).hexdigest()
print("estimates.csv SHA-256:", sha)
print("frozen hash matches:", sha == frozen)
if sha != frozen:
    print("Hash differs: likely other library versions writing floats differently "
          "(see notes/phase3_freeze.md). Compare values before grading.")
est = pd.read_csv(EST)
print(f"{len(est):,} estimate rows, {est.groupby(['lob','GRCODE']).ngroups} lines")

estimates.csv SHA-256: 68968d57fd202ebd0c1ee8d764e08a0372dc8ab2c332833b84db738e167c2a2f
frozen hash matches: True
62,320 estimate rows, 169 lines


## 7.2 Actuals at 120 months
From the full clean data (known and hidden cells): actual paid and reported at lag 10 for every company-AY, actual unpaid on each basis, and the case still open at 120 (reported − paid at 120).

In [2]:
# Step 7.2: actual losses at 120, both bases (reads hidden cells)
clean = bt.load_clean()
act = []
for lob, df in clean.items():
    a = df[df.DevelopmentLag == m.TARGET_LAG].assign(lob=lob)
    act.append(pd.DataFrame({"lob": lob, "GRCODE": a.GRCODE, "AccidentYear": a.AccidentYear,
                             "paid_120": a.CumPaidLoss, "reported_120": a.IncurredLosses - a.BulkLoss}))
act = pd.concat(act, ignore_index=True)
assert not act.duplicated(["lob", "GRCODE", "AccidentYear"]).any()

company_ays = est.drop_duplicates(["lob", "GRCODE", "AccidentYear"])[["lob", "GRCODE", "AccidentYear"]]
missing = company_ays.merge(act, how="left", indicator=True).query("_merge == 'left_only'")
assert missing.empty, f"{len(missing)} company-AYs have no lag 10 row"
act["open_case_120"] = act.reported_120 - act.paid_120

long_act = act.melt(id_vars=["lob", "GRCODE", "AccidentYear"], value_vars=["paid_120", "reported_120"],
                    var_name="basis", value_name="actual_120")
long_act["basis"] = long_act.basis.str.replace("_120", "")
print("company-AYs with actuals:", len(act), "| negative open case at 120 (≤ −2):",
      int((act.open_case_120 <= -2).sum()))

company-AYs with actuals: 1558 | negative open case at 120 (≤ −2): 29


## 7.3 AY-level grades
One row per company-AY × combination, AYs 1999–2007. `error` = estimate − actual at 120 ($000); `pct_ep` = error ÷ EP; `pct_reserve` = error ÷ actual unpaid where actual unpaid > 0 (kept for reference; AY-level results use % of EP). `age` is the maturity at 12/31/2007 for the Phase 4 maturity question.

In [3]:
# Step 7.3: grade every estimate, AY level
g = est[est.AccidentYear.isin(GRADED)].merge(long_act, on=["lob", "GRCODE", "AccidentYear", "basis"],
                                              how="left", validate="m:1")
assert g.actual_120.notna().all()
g["actual_unpaid"] = g.actual_120 - g.paid
g["error"] = g.est - g.actual_120
finite = np.isfinite(g.est)
g.loc[~finite, "error"] = np.nan
g["pct_ep"] = g.error / g.ep
g["pct_reserve"] = g.error / g.actual_unpaid.where(g.actual_unpaid > 0)
g.to_csv(RESULTS / "graded_ay.csv", index=False)
print(f"{len(g):,} graded rows ({len(g) // 40:,} company-AYs × 40); "
      f"non-finite estimates: {int((~finite).sum())}")

55,560 graded rows (1,389 company-AYs × 40); non-finite estimates: 96


## 7.4 Line-level totals
One row per line × combination: AYs 1999–2007 summed. These are the units the summary and the Phase 4 paired comparisons use (keyed on `lob, GRCODE, basis, method, average, elr`). A line with a non-finite estimate on any graded AY gets no total for that combination. `in_common` marks the D4 common sample.

In [4]:
# Step 7.4: line totals
KEYS = ["lob", "GRCODE", "basis", "method", "average", "elr"]
t = (g.assign(nonfinite=~np.isfinite(g.est))
       .groupby(KEYS, dropna=False)
       .agg(est_unpaid=("unpaid", "sum"), actual_unpaid=("actual_unpaid", "sum"),
            error=("error", "sum"), ep=("ep", "sum"), n_ays=("AccidentYear", "size"),
            nonfinite=("nonfinite", "any"))
       .reset_index())
t.loc[t.nonfinite, ["est_unpaid", "error"]] = np.nan
t["pct_reserve"] = t.error / t.actual_unpaid.where(t.actual_unpaid > 0)
t["pct_ep"] = t.error / t.ep

# D4: drop a line from head-to-head comparisons if any of its 40 combinations is non-finite
bad_lines = t[t.nonfinite][["lob", "GRCODE"]].drop_duplicates()
t["in_common"] = ~t.set_index(["lob", "GRCODE"]).index.isin(bad_lines.set_index(["lob", "GRCODE"]).index)

attrs = est.drop_duplicates(["lob", "GRCODE", "basis"])[
    ["lob", "GRCODE", "basis", "size_ep", "n_ays", "incomplete", "single", "ceded_swing", "zero_company_elr"]
].rename(columns={"n_ays": "n_ays_present"})
t = t.merge(attrs, on=["lob", "GRCODE", "basis"], how="left", validate="m:1")
t.to_csv(RESULTS / "graded_lines.csv", index=False)

lines = t.drop_duplicates(["lob", "GRCODE", "basis"])
print("lines left out of the common sample (D4):", bad_lines.values.tolist())
print("\nlines with actual unpaid total ≤ 0 (no % of reserve), by lob × basis:")
print(lines.assign(no_res=lines.actual_unpaid <= 0).groupby(["lob", "basis"]).no_res.agg(["sum", "size"]))

lines left out of the common sample (D4): [['WC', 26956]]

lines with actual unpaid total ≤ 0 (no % of reserve), by lob × basis:
              sum  size
lob basis              
PPA paid        8   109
    reported    8   109
WC  paid        3    60
    reported    1    60


## 7.5 Summary results table (step 8 checkpoint)
Per line of business × basis × method × average × ELR, over line totals: n, median signed error, 25th and 75th percentiles (the IQR), and median absolute error, on both scales. `sample = common` is the D4 common sample used for comparing methods; `own` is every line where that combination is finite. Saved to `docs/phase3_summary.csv` (committed, D5).

Medians, not means: a few chain ladder blowups on small lines move a mean, not a median (outlier flag cut for this reason).

In [5]:
# Step 7.5: summary table
def summarize(df):
    r = df.pct_reserve.dropna(); e = df.pct_ep.dropna()
    return pd.Series({
        "n": int(df.error.notna().sum()), "n_reserve": len(r),
        "med_res": r.median(), "p25_res": r.quantile(.25), "p75_res": r.quantile(.75),
        "med_abs_res": r.abs().median(),
        "med_ep": e.median(), "p25_ep": e.quantile(.25), "p75_ep": e.quantile(.75),
        "med_abs_ep": e.abs().median()})

KEY_COLS = ["lob", "basis", "method", "average", "elr"]
summary = pd.concat([
    t[t.in_common].groupby(KEY_COLS, dropna=False).apply(summarize, include_groups=False)
      .reset_index().assign(sample="common"),
    t[t.error.notna()].groupby(KEY_COLS, dropna=False).apply(summarize, include_groups=False)
      .reset_index().assign(sample="own"),
], ignore_index=True)
summary["n"] = summary.n.astype(int); summary["n_reserve"] = summary.n_reserve.astype(int)
assert len(summary) == 2 * 2 * 40, len(summary)
summary.to_csv(DOCS / "phase3_summary.csv", index=False, float_format="%.6f")

def label(r):
    if r.method == "cl": return f"CL {r.average}"
    if r.method == "ec": return f"EC {r.elr}"
    return f"BF {r.elr} {r.average}"
summary["combo"] = summary.apply(label, axis=1)
fmt = {c: "{:+.1%}".format for c in ["med_res", "p25_res", "p75_res", "med_ep", "p25_ep", "p75_ep"]}
fmt |= {"med_abs_res": "{:.1%}".format, "med_abs_ep": "{:.1%}".format}
print(f"{len(summary)} summary rows -> docs/phase3_summary.csv")

160 summary rows -> docs/phase3_summary.csv


### Headline comparison (common sample, D3 average)
Chain ladder and BF with volume-weighted all-years factors; expected claims with each ELR. % of reserve on line totals, AYs 1999–2007.

In [6]:
head = summary[(summary["sample"] == "common")
               & ((summary.method == "ec") | (summary.average == bt.HEADLINE_AVERAGE))]
cols = ["n", "n_reserve", "med_res", "p25_res", "p75_res", "med_abs_res", "med_ep", "med_abs_ep"]
for (lob, basis), h in head.groupby(["lob", "basis"]):
    print(f"\n{lob} {basis}")
    print(h.set_index("combo")[cols].to_string(formatters=fmt))


PPA paid
                          n  n_reserve med_res p25_res p75_res med_abs_res med_ep med_abs_ep
combo                                                                                       
BF company volume_all   109        101   +7.8%   -8.7%  +32.7%       22.8%  +0.3%       1.1%
BF industry volume_all  109        101  +14.8%   -2.3%  +35.9%       22.3%  +0.7%       1.2%
CL volume_all           109        101   +6.9%   -9.7%  +31.7%       18.8%  +0.2%       0.9%
EC company              109        101  -18.8% -126.3%  +68.3%      103.4%  -2.0%       7.3%
EC industry             109        101  +42.4%  -40.1% +116.6%       95.7%  +2.4%       6.8%

PPA reported
                          n  n_reserve med_res p25_res p75_res med_abs_res med_ep med_abs_ep
combo                                                                                       
BF company volume_all   109        101   +4.5%   -6.1%  +21.9%       12.8%  +0.2%       0.9%
BF industry volume_all  109        101   +3.9%

### Secondary: chain ladder under each of the 6 averages (common sample)

In [7]:
sec = summary[(summary["sample"] == "common") & (summary.method == "cl")]
for (lob, basis), h in sec.groupby(["lob", "basis"]):
    print(f"\n{lob} {basis}")
    print(h.set_index("combo")[cols].to_string(formatters=fmt))


PPA paid
                 n  n_reserve med_res p25_res p75_res med_abs_res med_ep med_abs_ep
combo                                                                              
CL simple_3    109        101   +3.6%  -11.2%  +23.2%       15.4%  +0.1%       0.9%
CL simple_5    109        101   +6.1%   -9.7%  +26.9%       17.1%  +0.2%       0.8%
CL simple_all  109        101   +8.1%   -9.5%  +35.1%       20.4%  +0.4%       1.1%
CL volume_3    109        101   +3.9%  -12.0%  +23.0%       16.0%  +0.1%       0.9%
CL volume_5    109        101   +4.9%  -10.5%  +25.7%       16.8%  +0.1%       0.9%
CL volume_all  109        101   +6.9%   -9.7%  +31.7%       18.8%  +0.2%       0.9%

PPA reported
                 n  n_reserve med_res p25_res p75_res med_abs_res med_ep med_abs_ep
combo                                                                              
CL simple_3    109        101   +0.2%   -7.6%   +9.3%        8.8%  +0.0%       0.6%
CL simple_5    109        101   +1.1%   -8.8%  +18.0

## 7.6 Case still open at 120 months, by line (step 8)
Reported-basis unpaid includes case still open at 120, so part of any paid vs reported gap is this built-in difference, not estimation error. Per line: open case at 120 summed over AYs 1999–2007, in $000 and as % of the line's actual reported unpaid. Saved to `docs/phase3_open_case_by_line.csv`.

In [8]:
oc = act[act.AccidentYear.isin(GRADED)].groupby(["lob", "GRCODE"]).agg(
    open_case_120=("open_case_120", "sum"), reported_120=("reported_120", "sum")).reset_index()
rep_unpaid = lines[lines.basis == "reported"][["lob", "GRCODE", "actual_unpaid"]]
oc = oc.merge(rep_unpaid.rename(columns={"actual_unpaid": "actual_reported_unpaid"}), on=["lob", "GRCODE"])
oc["pct_of_reported_unpaid"] = oc.open_case_120 / oc.actual_reported_unpaid.where(oc.actual_reported_unpaid > 0)
oc["pct_of_reported_120"] = oc.open_case_120 / oc.reported_120.where(oc.reported_120 > 0)
oc.to_csv(DOCS / "phase3_open_case_by_line.csv", index=False, float_format="%.6f")
print(oc.groupby("lob")[["open_case_120", "pct_of_reported_unpaid", "pct_of_reported_120"]]
        .describe(percentiles=[.25, .5, .75]).T.round(3))

lob                                  PPA          WC
open_case_120          count     109.000      60.000
                       mean     3495.982   18307.083
                       std     28061.512   55257.852
                       min      -117.000     -79.000
                       25%         0.000     101.500
                       50%        25.000     972.000
                       75%       227.000    5334.500
                       max    291775.000  312883.000
pct_of_reported_unpaid count     101.000      59.000
                       mean        0.045       0.225
                       std         0.155       0.271
                       min        -0.113      -0.099
                       25%         0.001       0.083
                       50%         0.005       0.151
                       75%         0.021       0.258
                       max         0.986       1.612
pct_of_reported_120    count     109.000      60.000
                       mean        0.004      